In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
## reading words
words = open('names.txt','r').read().splitlines()
words[:8]


['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
## creating characters vocabulary
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [10]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
def build_dataset(words):
    X, Y= [], []
    for w in words: 

        #print(w)
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            #print("'.join(itos[i] for i in context), '->', itos[ix])
            context = context[1:] + [ix] # crop and append

    X = torch. tensor(X)
    Y = torch. tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random. seed(42)
random. shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr, Ytr = build_dataset(words[:n1])            ## 80% for training
Xdev, Ydev = build_dataset(words[n1:n2])        ## 10% for development/validation
Xte, Yte = build_dataset(words[n2:])            ## 10% for testing

torch.Size([182580, 3]) torch.Size([182580])
torch.Size([22767, 3]) torch.Size([22767])
torch.Size([22799, 3]) torch.Size([22799])


In [11]:
## Boilerplate code is done
# function which will used to compare the gradient calculated manually and calculated by pytorch
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt,t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact : {str(ex):5s} | approximate : {str(app):5s} maxdiff : {maxdiff}')

In [12]:
n_embed = 10
n_hidden = 64

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embed), generator=g)

w1 = torch.randn((n_embed * block_size, n_hidden), generator = g) * (5/3)/ (n_embed * block_size)**0.5
b1 = torch.randn((n_hidden), generator = g) * 0.1
w2 = torch.randn((n_hidden, vocab_size), generator = g) * 0.1
b2 = torch.randn((vocab_size), generator = g) * 0.1

## batch norm parameters
gain = torch.randn((1, n_hidden)) * 0.1 + 1.0
shift = torch.randn((1, n_hidden)) * 0.1

parameters = [C,w1,b1,w2,b2,gain,shift]
length = sum(p.nelement() for p in parameters)
for p in parameters:
    p.requires_grad = True

In [16]:
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]  # batch X,Y

In [19]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb]  # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1)  # concatenate the vectors

# Linear layer 1
hprebn = embcat @ w1 + b1  # hidden layer pre-activation

# BatchNorm layer
bnmeani = 1 / n * hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1 / (n - 1) * (bndiff2).sum(0, keepdim=True)  # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = gain * bnraw + shift

# Non-linearity
h = torch.tanh(hpreact)  # hidden layer

# Linear layer 2
logits = h @ w2 + b2  # output layer

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes  # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1  # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact.
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
    p.grad = None

for t in [
    logprobs, probs, counts, counts_sum, counts_sum_inv,
    norm_logits, logit_maxes, logits, h, hpreact, bnraw,
    bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
    embcat, emb
]:
    t.retain_grad()

loss.backward()

loss

tensor(3.4261, grad_fn=<NegBackward0>)

In [141]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
dprobs = (1/probs) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(dim=1, keepdim=True)
dcounts = counts_sum_inv * dprobs
dcounts_sum = dcounts_sum_inv * -1*counts_sum**-2
dcounts += torch.ones_like(counts) * dcounts_sum
dnorm_logits = dcounts * norm_logits.exp()
dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
dlogits = dnorm_logits + F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

dh = dlogits @ w2.T
dw2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = dh * (1-h**2)
dgain = (dhpreact * bnraw).sum(0, keepdim=True)
dshift = dhpreact.sum(0, keepdim=True)
dbnraw = (dhpreact * gain)
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = dbnvar_inv * (-0.5*(bnvar + 1e-5)**(-1.5))
dbndiff2 = dbnvar * (1/(n-1)) * torch.ones_like(bndiff2)
dbndiff = bnvar_inv * dbnraw
dbndiff += dbndiff2 * 2 * bndiff
dbnmeani = (-dbndiff).sum(0, keepdim=True)
dhprebn = dbndiff
dhprebn += 1/n * torch.ones_like(bnmeani) * dbnmeani
dembcat = dhprebn @ w1.T
dw1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(batch_size, block_size, n_embed)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k,j]
        dC[ix] += demb[k,j]
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dw2, w2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dgain, gain)
cmp('bnbias', dshift, shift) 
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv) 
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dw1, w1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact : True  | approximate : True  maxdiff : 0.0
probs           | exact : True  | approximate : True  maxdiff : 0.0
counts_sum_inv  | exact : True  | approximate : True  maxdiff : 0.0
counts_sum      | exact : True  | approximate : True  maxdiff : 0.0
counts          | exact : True  | approximate : True  maxdiff : 0.0
norm_logits     | exact : True  | approximate : True  maxdiff : 0.0
logit_maxes     | exact : True  | approximate : True  maxdiff : 0.0
logits          | exact : True  | approximate : True  maxdiff : 0.0
h               | exact : True  | approximate : True  maxdiff : 0.0
W2              | exact : True  | approximate : True  maxdiff : 0.0
b2              | exact : True  | approximate : True  maxdiff : 0.0
hpreact         | exact : True  | approximate : True  maxdiff : 0.0
bngain          | exact : True  | approximate : True  maxdiff : 0.0
bnbias          | exact : True  | approximate : True  maxdiff : 0.0
bnraw           | exact : True  | approximate : 

In [70]:
print(f'h shape:{h.shape}') , print(f'w2 shape:{w2.shape}') , print(f'b2 shape:{b2.shape}') , print(f'dlogits shape:{dlogits.shape}')

h shape:torch.Size([32, 64])
w2 shape:torch.Size([64, 27])
b2 shape:torch.Size([27])
dlogits shape:torch.Size([32, 27])


(None, None, None, None)

In [142]:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.426133155822754 diff: 0.0


In [ ]:
# backward pass
dlogits = F.softmax(logits,1)
dlogits[range(n), Yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

logits          | exact : False | approximate : True  maxdiff : 4.889443516731262e-09
